# MorphiNet — Source ↔ Anchor Orientation Helper

Re-orient a **source dataset** so its heart segmentation matches the orientation of a known-good
**anchor** volume (`template/acdc_anchor_label.nii.gz`). The anchor is already in the orientation
MorphiNet's pipeline expects, so matching the source to it guarantees the source will work too —
**no template mesh required**.

**Workflow**
1. Point `SOURCE_DIR` at your unchanged data (MorphiNet label convention, `imagesTs/labelsTs`).
2. In the **interactive 3D view**, flip/swap the source until it overlays the anchor — drag to
   rotate, scroll to zoom. Each token you apply re-renders the view.
3. Lock the sequence, then set `TARGET_PIXDIM` (you confirm the source spacing in the anchor's axis
   order — the tool does not guess it), and bake: every source subject is re-ordered in **pixel
   space** and written with a fresh **anchor-aligned affine** — the anchor's orientation carrying
   `TARGET_PIXDIM`, origin at the voxel corner (dtypes preserved) — to `OUTPUT_DIR/imagesTs|labelsTs`
   in MorphiNet convention.

Transforms are flip/swap signed-permutations reusing `data/utils/geometry.py` — the same vocabulary
`data.components.SequentialTransformd` applies at load time. The baked affine is rebuilt from the
anchor rather than compensated from the source, so the world location is intentionally not preserved.

In [ ]:
# @cell imports
import os, sys, glob, json, datetime, tempfile
from pathlib import Path
import numpy as np
import nibabel as nib
import torch
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# Make the MorphiNet package importable regardless of where Jupyter was started.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data" / "utils" / "geometry.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not locate MorphiNet project root (data/utils/geometry.py).")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from data.utils.geometry import (
    parse_transform_sequence, compose_sequence_matrix, apply_tensor_sequence,
)
from utils.path_config import get_path_default

print("project root:", PROJECT_ROOT)

In [ ]:
# @cell config
# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# SOURCE_DIR: your unchanged data to re-orient (MorphiNet label convention).
#   Layout: SOURCE_DIR/<IMAGES_SUBDIR>/<stem><IMAGE_SUFFIX>
#           SOURCE_DIR/<LABELS_SUBDIR>/<stem>.nii.gz
SOURCE_DIR = Path("/mnt/data/Experiment/Data/MorphiNet-MR_CT/Archive/Dataset021_ACDC")
IMAGES_SUBDIR, LABELS_SUBDIR = "imagesTs", "labelsTs"
IMAGE_SUFFIX = "_0000.nii.gz"   # image files; label = same stem without this suffix + ".nii.gz"

# OUTPUT_DIR: where the re-oriented, pipeline-ready tree is written (config.env target by default).
_acdc_default = get_path_default("MORPHINET_ACDC_DATA_DIR")
OUTPUT_DIR = Path(_acdc_default) if os.path.isabs(_acdc_default) else (PROJECT_ROOT / _acdc_default)

# ANCHOR: the reference orientation (one subject shipped under template/).
ANCHOR_LABEL = PROJECT_ROOT / "template" / "acdc_anchor_label.nii.gz"

# MorphiNet label convention in the volumes.
MYO_LABELS = (2, 4)   # lv-myo + rv-myo == bi-ventricular myocardium
RV_CAVITY  = 3
LV_CAVITY  = 1

print("source:", SOURCE_DIR)
print("output:", OUTPUT_DIR)
print("anchor:", ANCHOR_LABEL, "| exists:", ANCHOR_LABEL.exists())

In [ ]:
# @cell core-fns
def load_nifti(path):
    o = nib.load(str(path))
    return np.asarray(o.dataobj), o.affine.copy(), o.header.copy()

def parse_seq(seq):
    seq = (seq or "").strip()
    return parse_transform_sequence(seq) if seq else []

def apply_sequence_to_array(arr, steps):
    '''Apply flip/swap steps to a 3D (H,W,D) array via the pipeline tensor op.'''
    if not steps:
        return np.ascontiguousarray(arr)
    t = torch.from_numpy(np.ascontiguousarray(arr))
    t = apply_tensor_sequence(t, steps)
    return np.ascontiguousarray(t.numpy())

def vol_to_xyz(coords_hwd, shape):
    '''Map (h,w,d) voxel coords to project (x,y,z) unit coords in [-1,1].
    Axis convention matches data/utils/geometry (the flip/swap sequence notation):
    x <-> D (dim 2), y <-> W (dim 1), z <-> H (dim 0). Same mapping for source and anchor.'''
    H, W, D = shape[-3:]
    out = np.empty((coords_hwd.shape[0], 3), dtype=float)
    out[:, 0] = 2.0 * coords_hwd[:, 2] / max(D - 1, 1) - 1.0   # x <- D (dim 2)
    out[:, 1] = 2.0 * coords_hwd[:, 1] / max(W - 1, 1) - 1.0   # y <- W (dim 1)
    out[:, 2] = 2.0 * coords_hwd[:, 0] / max(H - 1, 1) - 1.0   # z <- H (dim 0)
    return out

def myo_points(label_arr, myo_labels=MYO_LABELS, max_pts=3500):
    coords = np.argwhere(np.isin(label_arr, list(myo_labels)))
    if coords.shape[0] == 0:
        return None
    if coords.shape[0] > max_pts:
        coords = coords[np.linspace(0, coords.shape[0] - 1, max_pts).astype(int)]
    return vol_to_xyz(coords.astype(float), label_arr.shape)

def cavity_centroid(label_arr, labels):
    coords = np.argwhere(np.isin(label_arr, list(labels)))
    if coords.shape[0] == 0:
        return None
    return vol_to_xyz(coords.mean(0, keepdims=True).astype(float), label_arr.shape)[0]

def list_source_stems(src_dir):
    return [p.name[:-len(IMAGE_SUFFIX)]
            for p in sorted((Path(src_dir) / IMAGES_SUBDIR).glob("*" + IMAGE_SUFFIX))]

def load_source_label(src_dir, stem):
    return load_nifti(Path(src_dir) / LABELS_SUBDIR / (stem + ".nii.gz"))

In [ ]:
# @cell overlay
# Interactive overlay using a plain plotly go.Figure (renders reliably in VS Code, JupyterLab and
# classic Notebook): anchor (fixed) vs source (after a flip/swap sequence), compared in a shared
# per-axis-normalized (x,y,z) frame (x<->D, y<->W, z<->H; see vol_to_xyz).
def _xyz(coords):
    if coords is None:
        return [], [], []
    a = np.atleast_2d(np.asarray(coords, dtype=float))
    return a[:, 0].tolist(), a[:, 1].tolist(), a[:, 2].tolist()

def _markers(coords, name, color, size, symbol="circle", opacity=1.0):
    x, y, z = _xyz(coords)
    return go.Scatter3d(x=x, y=y, z=z, mode="markers", name=name,
                        marker=dict(size=size, color=color, symbol=symbol, opacity=opacity))

def _traces(anchor_lbl, source_lbl, steps):
    src = apply_sequence_to_array(source_lbl, steps)
    return [
        _markers(myo_points(anchor_lbl),                   "anchor myo", "gray",   3, opacity=0.45),
        _markers(myo_points(src),                          "source myo", "orange", 3, opacity=0.7),
        _markers(cavity_centroid(anchor_lbl, [RV_CAVITY]), "anchor RV",  "red",   10, "diamond"),
        _markers(cavity_centroid(anchor_lbl, [LV_CAVITY]), "anchor LV",  "green", 10, "diamond"),
        _markers(cavity_centroid(src, [RV_CAVITY]),        "source RV",  "blue",  12, "x"),
        _markers(cavity_centroid(src, [LV_CAVITY]),        "source LV",  "cyan",  12, "x"),
    ]

# Light axis frame: enough ticks/grid/zero-lines + x/y/z titles to read orientation while rotating,
# but no dense default ticks and no colored background panes.
_AXIS = dict(showgrid=True, gridcolor="rgba(0,0,0,0.12)", nticks=4,
             showticklabels=True, tickfont=dict(size=9),
             zeroline=True, zerolinecolor="rgba(0,0,0,0.35)",
             showbackground=False, showspikes=False, range=[-1.1, 1.1])
def _layout(title=""):
    return dict(title=title, height=600, margin=dict(l=0, r=0, t=40, b=0),
                legend=dict(orientation="h", yanchor="bottom", y=0.0, x=0.0),
                scene=dict(aspectmode="cube",
                           camera=dict(eye=dict(x=1.5, y=1.5, z=1.2)),
                           xaxis=dict(title="x", **_AXIS),
                           yaxis=dict(title="y", **_AXIS),
                           zaxis=dict(title="z", **_AXIS)))

def compare_figure(anchor_lbl, source_lbl, steps, title=""):
    '''Interactive overlay figure (drag to rotate, scroll to zoom).'''
    f = go.Figure(_traces(anchor_lbl, source_lbl, steps))
    f.update_layout(**_layout(title))
    return f

anchor_label_arr, anchor_affine, _ = load_nifti(ANCHOR_LABEL)
SOURCE_STEMS = list_source_stems(SOURCE_DIR)
assert SOURCE_STEMS, f"no '*{IMAGE_SUFFIX}' under {SOURCE_DIR / IMAGES_SUBDIR}"
_src0, _, _ = load_source_label(SOURCE_DIR, SOURCE_STEMS[0])
print("anchor", tuple(anchor_label_arr.shape), "| source subjects:", len(SOURCE_STEMS),
      "| source[0]", SOURCE_STEMS[0], tuple(_src0.shape))

# Standalone sanity view — renders directly in this cell's output (independent of the widgets
# below). If you can see the gray anchor cloud + red/green diamonds here and rotate it, rendering
# works. Source identity is orange + blue/cyan crosses.
compare_figure(anchor_label_arr, _src0, [], title=f"{SOURCE_STEMS[0]}: identity vs anchor").show()

## Align the source to the anchor

**Drag** to rotate, **scroll** to zoom. Pick a subject, then click flip/swap buttons; the view
**re-renders** under the controls each time. Aim to land the source markers on the anchor markers —
blue ✕ on red ◆ (RV), cyan ✕ on green ◆ (LV) — with the orange myo cloud over the gray one.
`s:` swaps reorder axes, `f:` flips mirror an axis (**order matters**). The plot's **x, y, z** axes
use the same notation as the sequence (project convention: `x`↔`D`, `y`↔`W`, `z`↔`H` array dims), so
applying `f:x` flips along the plot's x-axis. Spot-check a couple of subjects: the right sequence works for all.

> The view resets to the default camera on each apply (a plain `go.Figure` re-render — the most
> portable option across notebook frontends). Rotate freely to inspect after each token.

In [ ]:
# @cell ui
seq_state = {"seq": ""}
subject_dd = widgets.Dropdown(options=SOURCE_STEMS, value=SOURCE_STEMS[0], description="subject:")
seq_text = widgets.Text(value="", description="sequence:", layout=widgets.Layout(width="55%"))
status = widgets.HTML()
view_out = widgets.Output()

# capture(clear_output=True) makes each draw REPLACE the previous figure in view_out, so repeated
# applies don't stack copies of the plot in the output.
@view_out.capture(clear_output=True, wait=True)
def _draw(src_lbl, steps, title):
    compare_figure(anchor_label_arr, src_lbl, steps, title=title).show()

def refresh(*_):
    try:
        steps = parse_seq(seq_text.value)
    except Exception as e:
        status.value = f"<b style='color:crimson'>invalid sequence: {e}</b>"
        view_out.clear_output()
        return
    seq_state["seq"] = seq_text.value.strip()
    src_lbl, _, _ = load_source_label(SOURCE_DIR, subject_dd.value)
    shp = tuple(apply_sequence_to_array(src_lbl, steps).shape)
    status.value = (f"subject <b>{subject_dd.value}</b> &nbsp; sequence <b>{seq_state['seq'] or '(identity)'}</b>"
                    f" &nbsp; source {tuple(src_lbl.shape)} &rarr; {shp} &nbsp; anchor {tuple(anchor_label_arr.shape)}")
    _draw(src_lbl, steps, f"{subject_dd.value}: {seq_state['seq'] or 'identity'}")

def _append(tok): seq_text.value = (seq_text.value + " " + tok).strip()
def _clear():     seq_text.value = ""
def _undo():      seq_text.value = " ".join(seq_text.value.split()[:-1])

token_buttons = []
for tok in ["f:x", "f:y", "f:z", "s:xy", "s:xz", "s:yz"]:
    b = widgets.Button(description=tok, layout=widgets.Layout(width="62px"))
    b.on_click(lambda _b, t=tok: _append(t))
    token_buttons.append(b)
undo_btn = widgets.Button(description="undo", layout=widgets.Layout(width="62px"))
clear_btn = widgets.Button(description="clear", button_style="warning", layout=widgets.Layout(width="62px"))
undo_btn.on_click(lambda _b: _undo())
clear_btn.on_click(lambda _b: _clear())
seq_text.observe(lambda ch: refresh() if ch["name"] == "value" else None, names="value")
subject_dd.observe(lambda ch: refresh() if ch["name"] == "value" else None, names="value")

display(widgets.VBox([subject_dd, widgets.HBox(token_buttons + [undo_btn, clear_btn]), seq_text, status]))
display(view_out)
refresh()

## Lock the sequence

Set `FINAL_SEQUENCE` to the string you settled on (copy it from the box above). The lock cell prints
the **anchor** and **source** voxel spacing (`pixdim`) and asks **you** to set `TARGET_PIXDIM` — the
tool does **not** guess the axis mapping for you.

**Target affine recipe** — for each subject the baked NIfTI gets a fresh affine:
`target[:3,:3] = normalize(anchor_affine[:3,:3]) @ diag(TARGET_PIXDIM)`, translation `0`.
Set `TARGET_PIXDIM` to the **source** spacing reordered into the **anchor's** `(dx, dy, dz)` order:
the two in-plane values first, the distinct through-plane value last (matching the anchor's layout).

In [ ]:
# @cell lock-validate
FINAL_SEQUENCE = seq_state.get("seq", "").strip()   # or hard-code, e.g. "s:xz f:x f:y" (ACDC archive->anchor)
steps = parse_seq(FINAL_SEQUENCE)
print("FINAL_SEQUENCE:", FINAL_SEQUENCE or "(identity)")

# --- Target affine helpers ---------------------------------------------------
# The baked target carries the ANCHOR's orientation with a voxel spacing YOU set
# (TARGET_PIXDIM), origin at the voxel corner (translation 0). World location is
# not preserved -- we rebake a clean, anchor-aligned affine, we do not compensate
# the source's original one.
#
#   target[:3, :3] = normalize(anchor_affine[:3, :3]) @ diag(TARGET_PIXDIM)
#   target[:3,  3] = 0

def anchor_orientation(anchor_affine):
    '''Unit-column direction cosines of the anchor (voxel spacing stripped).'''
    M = np.asarray(anchor_affine, float)[:3, :3]
    return M / np.linalg.norm(M, axis=0)

def read_pixdim(path):
    '''Voxel spacing (3,) straight from the NIfTI header.'''
    return np.asarray(nib.load(str(path)).header["pixdim"][1:4], float)

def make_target_affine(anchor_affine, target_pixdim):
    '''4x4 affine = anchor orientation x diag(target_pixdim), zero translation.'''
    if target_pixdim is None:
        raise ValueError("TARGET_PIXDIM is not set -- set it in the lock-validate cell.")
    A = np.eye(4)
    A[:3, :3] = anchor_orientation(anchor_affine) @ np.diag(np.asarray(target_pixdim, float))
    return A

# --- Voxel spacing: YOU set the target (the tool does NOT guess) -------------
# Compare the two spacings printed below, then set TARGET_PIXDIM yourself to the
# SOURCE spacing reordered into the ANCHOR's axis order: the two in-plane values
# (dx, dy) first and the distinct through-plane value (dz) LAST -- the anchor's
# layout. Example: source header [10.0, 1.5625, 1.5625] and anchor
# [1.5625, 1.5625, 10.0]  ->  TARGET_PIXDIM = [1.5625, 1.5625, 10.0].
anchor_pixdim = read_pixdim(ANCHOR_LABEL)
source_pixdim = read_pixdim(Path(SOURCE_DIR) / LABELS_SUBDIR / (SOURCE_STEMS[0] + ".nii.gz"))
print("anchor pixdim  (TARGET axis order: dx, dy, dz):", np.round(anchor_pixdim, 4).tolist())
print(f"source pixdim  (header order, {SOURCE_STEMS[0]}):", np.round(source_pixdim, 4).tolist())

# >>> SET THIS: source spacing in the anchor's (dx, dy, dz) order, e.g. [1.5625, 1.5625, 10.0]
TARGET_PIXDIM = None

if TARGET_PIXDIM is None:
    print("\n>>> ACTION REQUIRED: set TARGET_PIXDIM above to the source spacing in the anchor's")
    print(">>> (dx, dy, dz) order -- distinct through-plane value LAST -- then re-run this cell.")
else:
    src_lbl, _, _ = load_source_label(SOURCE_DIR, SOURCE_STEMS[0])
    target_aff = make_target_affine(anchor_affine, TARGET_PIXDIM)

    # (1) the pixel sequence is invertible (a signed permutation always is)
    P = compose_sequence_matrix(steps, src_lbl.shape) if steps else np.eye(4)
    assert np.allclose(P @ np.linalg.inv(P), np.eye(4)), "non-invertible transform"
    # (2) the baked affine carries the anchor's orientation and your chosen spacing
    assert np.allclose(anchor_orientation(target_aff), anchor_orientation(anchor_affine), atol=1e-6), \
        "target orientation != anchor orientation"
    assert np.allclose(np.linalg.norm(target_aff[:3, :3], axis=0), np.asarray(TARGET_PIXDIM, float), atol=1e-4), \
        "target spacing != TARGET_PIXDIM"
    if not np.allclose(sorted(source_pixdim), sorted(np.asarray(TARGET_PIXDIM, float)), atol=1e-3):
        print(f"  NOTE: TARGET_PIXDIM {list(TARGET_PIXDIM)} is not a reordering of the source "
              f"spacing {np.round(source_pixdim, 4).tolist()} -- double-check it is intended.")

    new_lbl = apply_sequence_to_array(src_lbl, steps)
    print("TARGET_PIXDIM:", list(TARGET_PIXDIM))
    print("target affine:\n", np.round(target_aff, 4))
    print("checks passed |", f"{SOURCE_STEMS[0]}: source {tuple(src_lbl.shape)} -> {tuple(new_lbl.shape)}"
          f" | anchor {tuple(anchor_label_arr.shape)}")
    compare_figure(anchor_label_arr, src_lbl, steps, title="FINAL: " + (FINAL_SEQUENCE or "identity")).show()

## Bake the whole source dataset

Applies `FINAL_SEQUENCE` to every `*_0000.nii.gz` image and its label (pixel-space re-order) and
writes the re-oriented tree to `OUTPUT_DIR/imagesTs|labelsTs` with a fresh **anchor-aligned affine**
(`anchor orientation × diag(TARGET_PIXDIM)`, dtypes preserved). Emits an
`orientation_preprocessing_report.json` (sequence, `TARGET_PIXDIM`, target affine, and per-subject
source pixdim; flags any subject whose spacing is not a reordering of `TARGET_PIXDIM`). Re-runnable
(overwrites). Requires `TARGET_PIXDIM` set in the lock cell. Try `limit=2` first.

In [ ]:
# @cell batch
def process_one(img_path, lbl_path, steps, target_pixdim, out_img, out_lbl):
    img_arr, _, _ = load_nifti(img_path)
    img_pd = read_pixdim(img_path)
    rec = {"file": os.path.basename(str(img_path)), "original_shape": list(img_arr.shape),
           "source_pixdim": [round(float(v), 4) for v in img_pd]}
    aff = make_target_affine(anchor_affine, target_pixdim)   # same orientation+spacing for img and label
    new_img = apply_sequence_to_array(img_arr, steps)
    nib.save(nib.Nifti1Image(new_img.astype(img_arr.dtype), aff), str(out_img))
    rec["transformed_shape"] = list(new_img.shape)
    warns = []
    if not np.allclose(sorted(img_pd), sorted(np.asarray(target_pixdim, float)), atol=1e-3):
        warns.append(f"source pixdim {np.round(img_pd, 4).tolist()} is not a reordering of "
                     f"TARGET_PIXDIM {[round(float(v), 4) for v in target_pixdim]}")
    if lbl_path is not None and os.path.exists(str(lbl_path)):
        lab_arr, _, _ = load_nifti(lbl_path)
        new_lab = apply_sequence_to_array(lab_arr, steps)
        nib.save(nib.Nifti1Image(new_lab.astype(lab_arr.dtype), aff), str(out_lbl))
        if not np.isin(new_lab, list(MYO_LABELS)).any():
            warns.append("no myocardium voxels after transform")
    else:
        warns.append("label missing")
    rec["warning"] = "; ".join(warns) if warns else None
    return rec

def process_dataset(src_dir, out_dir, sequence, target_pixdim, limit=None):
    steps = parse_seq(sequence)
    src_dir, out_dir = Path(src_dir), Path(out_dir)
    (out_dir / IMAGES_SUBDIR).mkdir(parents=True, exist_ok=True)
    (out_dir / LABELS_SUBDIR).mkdir(parents=True, exist_ok=True)
    img_paths = sorted((src_dir / IMAGES_SUBDIR).glob("*" + IMAGE_SUFFIX))
    if limit:
        img_paths = img_paths[:limit]
    if not img_paths:
        raise FileNotFoundError(f"no '*{IMAGE_SUFFIX}' under {src_dir / IMAGES_SUBDIR}")
    records, warnings = [], []
    for i, ip in enumerate(img_paths):
        stem = ip.name[:-len(IMAGE_SUFFIX)]
        lp = src_dir / LABELS_SUBDIR / (stem + ".nii.gz")
        rec = process_one(ip, lp if lp.exists() else None, steps, target_pixdim,
                          out_dir / IMAGES_SUBDIR / ip.name,
                          out_dir / LABELS_SUBDIR / (stem + ".nii.gz"))
        records.append(rec)
        if rec["warning"]:
            warnings.append((rec["file"], rec["warning"]))
        if (i + 1) % 25 == 0 or i + 1 == len(img_paths):
            print(f"  {i+1}/{len(img_paths)} processed")
    report = {
        "timestamp": datetime.datetime.now().isoformat(),
        "source_dir": str(src_dir),
        "output_dir": str(out_dir),
        "anchor_label": str(ANCHOR_LABEL),
        "sequence": sequence or "(identity)",
        "target_pixdim": [round(float(v), 4) for v in target_pixdim],
        "pixel_permutation_matrix": (compose_sequence_matrix(steps, records[0]["original_shape"]).tolist()
                                     if steps else np.eye(4).tolist()),
        "target_affine": make_target_affine(anchor_affine, target_pixdim).tolist(),
        "total_processed": len(records),
        "total_warnings": len(warnings),
        "warnings": warnings,
        "shape_examples": records[:10],
    }
    with open(out_dir / "orientation_preprocessing_report.json", "w") as f:
        json.dump(report, f, indent=2)
    print(f"done: {len(records)} files -> {out_dir} ; warnings: {len(warnings)}")
    return report

# Smoke first (writes 2 files), then run the full bake:
# process_dataset(SOURCE_DIR, OUTPUT_DIR, FINAL_SEQUENCE, TARGET_PIXDIM, limit=2)
# report = process_dataset(SOURCE_DIR, OUTPUT_DIR, FINAL_SEQUENCE, TARGET_PIXDIM)

## Make it pipeline-ready

The baked tree carries the **anchor's orientation and the source's voxel spacing** (origin at the
voxel corner), so the pipeline processes it exactly like the anchor — no pipeline change is needed.
World location is not preserved, which is fine: MorphiNet aligns segmentations by orientation and
spacing, not absolute world position. To wire it up:

1. Point `MORPHINET_ACDC_DATA_DIR` (`config.env`) at `OUTPUT_DIR`.
2. The inference loader reads the **`test`** split of `dataset/dataset_task21_f0.json` from
   `MORPHINET_ACDC_DATA_DIR/imagesTs|labelsTs`; file names are preserved, so the existing split keeps
   working. The cell below regenerates a `test` split from the baked files as
   `OUTPUT_DIR/dataset_task21_f0.generated.json` for inspection.

In [ ]:
# @cell json-regen
def regenerate_test_split(out_dir, json_template_path=None):
    out_dir = Path(out_dir)
    imgs = sorted((out_dir / IMAGES_SUBDIR).glob("*" + IMAGE_SUFFIX))
    entries = []
    for ip in imgs:
        stem = ip.name[:-len(IMAGE_SUFFIX)]
        if (out_dir / LABELS_SUBDIR / (stem + ".nii.gz")).exists():
            entries.append({"image": f"./{IMAGES_SUBDIR}/{ip.name}",
                            "label": f"./{LABELS_SUBDIR}/{stem}.nii.gz"})
    doc = {}
    if json_template_path and Path(json_template_path).exists():
        with open(json_template_path) as f:
            doc = json.load(f)
    doc["test"], doc["numTest"] = entries, len(entries)
    gen = out_dir / "dataset_task21_f0.generated.json"
    with open(gen, "w") as f:
        json.dump(doc, f, indent=2)
    print(f"wrote {len(entries)} test entries -> {gen}")
    return gen

# regenerate_test_split(OUTPUT_DIR, PROJECT_ROOT / "dataset" / "dataset_task21_f0.json")

## Validate a baked subject

After baking, re-load one saved subject and overlay it on the anchor with **identity** — it should
already match (markers coincident, clouds overlapping). Empty-myo warnings in the report flag
subjects to inspect.

In [ ]:
# @cell validate-baked
def validate_baked(out_dir, stem):
    lbl, baked_aff, _ = load_nifti(Path(out_dir) / LABELS_SUBDIR / (stem + ".nii.gz"))
    print(stem, "baked shape:", tuple(lbl.shape), "labels:", np.unique(lbl).tolist())
    print("baked affine:\n", np.round(baked_aff, 4))
    print("orientation matches anchor:",
          np.allclose(anchor_orientation(baked_aff), anchor_orientation(anchor_affine), atol=1e-6))
    compare_figure(anchor_label_arr, lbl, [], title=f"baked {stem} vs anchor (identity)").show()

# after a bake, e.g.:
# validate_baked(OUTPUT_DIR, SOURCE_STEMS[0])